# Обучение NER для извлечения характеристик (SpecFlow)

Блокнот дообучает `ruBERT-base` на weak-разметке из `train.jsonl` и
`val.jsonl`: BIO-метки `CHARACTERISTIC` (имя характеристики) и
`PROPERTY` (значение).

## Как запустить

1. Загрузи `train.jsonl` и `val.jsonl` в Colab. Блокнот сам найдёт их рядом с
   собой. Если не найдёт — покажет диалог загрузки: выбери оба файла.
2. `Среда выполнения → Сменить тип среды выполнения → T4 GPU` (обязательно GPU).
3. Запусти ячейки по порядку (`Ctrl+F9` — запустить всё).

## Ожидаемое время

На T4: конвертация ~1 мин, обучение 2 эпохи ~40-70 мин, оценка ~3-5 мин.

## Что получится

Каталог модели: `config.json`, веса, токенизатор, `labels.json`, `metrics.json`.
Метрики — entity-level precision/recall/F1 по сущностям (не по токенам).


In [ ]:
import importlib
import os
import subprocess
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"

REQUIRED = {
    "torch": "torch",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "tqdm": "tqdm",
}


def ensure_packages(packages):
    missing = []
    for module, package in packages.items():
        try:
            importlib.import_module(module)
        except Exception:
            missing.append(package)
    if missing:
        print("Устанавливаю:", " ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Все зависимости уже установлены")


ensure_packages(REQUIRED)

import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, f"{props.total_memory / 1e9:.1f} ГБ")
else:
    print("ВНИМАНИЕ: GPU не найден. Обучение ruBERT-base на CPU будет очень медленным.")
    print("Смени среду выполнения на T4 GPU: Среда выполнения → Сменить тип среды выполнения.")


## 1. Данные

Нужны `train.jsonl` и `val.jsonl` (weak-датасет). Ноутбук ищет их в `/content`,
`/content/ner`, `/content/drive/MyDrive`, `/content/drive/MyDrive/ner`.
Если не находит — предложит загрузить вручную (можно zip).

Если хочешь сохранять чекпойнты и модель в Google Drive — выполни **до** обучения:

```python
from google.colab import drive
drive.mount('/content/drive')
```


In [ ]:
from pathlib import Path

TRAIN_NAME = "train.jsonl"
VAL_NAME = "val.jsonl"
SEARCH_DIRS = [Path("."), Path("/content")]


def find_data(name):
    for directory in SEARCH_DIRS:
        candidate = directory / name
        if candidate.exists():
            return candidate
    return None


train_path = find_data(TRAIN_NAME)
val_path = find_data(VAL_NAME)

if train_path is None or val_path is None:
    from google.colab import files

    print("Загрузи train.jsonl и val.jsonl (можно оба сразу или zip-архив):")
    uploaded = files.upload()
    for name in list(uploaded):
        if name.endswith(".zip"):
            import zipfile

            with zipfile.ZipFile(name) as archive:
                archive.extractall(".")
    train_path = find_data(TRAIN_NAME)
    val_path = find_data(VAL_NAME)

if train_path is None or val_path is None:
    raise FileNotFoundError("Не найдены train.jsonl и/или val.jsonl")

print("train:", train_path, f"{train_path.stat().st_size / 1e6:.1f} МБ")
print("val:", val_path, f"{val_path.stat().st_size / 1e6:.1f} МБ")


## 2. Weak-разметка → BIO

Запись weak-датасета: `{"text": ..., "spans": [{"start", "end", "label", "attribute_code"}], ...}`.

Текст режется по пробельным границам; слово, пересекающееся со спаном, считается
частью сущности. Первое слово — `B-`, следующие — `I-`. При пересечении
приоритет у `PROPERTY`. `attribute_code` для обучения не используется:
модель учится находить сущности, а не коды полей.


In [ ]:
import json
import re
from dataclasses import dataclass

from tqdm.auto import tqdm

NER_LABEL_CHARACTERISTIC = "CHARACTERISTIC"
NER_LABEL_PROPERTY = "PROPERTY"
NER_BIO_OUTSIDE = "O"
NER_BIO_PREFIX_BEGIN = "B-"
NER_BIO_PREFIX_INSIDE = "I-"

_WORD_PATTERN = re.compile(r"\S+")

LABELS = [
    NER_BIO_OUTSIDE,
    f"{NER_BIO_PREFIX_BEGIN}{NER_LABEL_CHARACTERISTIC}",
    f"{NER_BIO_PREFIX_INSIDE}{NER_LABEL_CHARACTERISTIC}",
    f"{NER_BIO_PREFIX_BEGIN}{NER_LABEL_PROPERTY}",
    f"{NER_BIO_PREFIX_INSIDE}{NER_LABEL_PROPERTY}",
]
LABEL2ID = {label: index for index, label in enumerate(LABELS)}
ID2LABEL = {index: label for index, label in enumerate(LABELS)}


@dataclass(frozen=True)
class WordSpan:
    text: str
    start: int
    end: int


@dataclass(frozen=True)
class BioExample:
    words: list
    labels: list


def tokenize_words(text):
    return [WordSpan(m.group(), m.start(), m.end()) for m in _WORD_PATTERN.finditer(text)]


def _overlap(word, start, end):
    return word.start < end and start < word.end


def align_labels(words, spans):
    labels = [NER_BIO_OUTSIDE] * len(words)
    ordered = sorted(
        spans,
        key=lambda span: 0 if span.get("label") == NER_LABEL_CHARACTERISTIC else 1,
    )
    for span in ordered:
        start = span.get("start")
        end = span.get("end")
        label = span.get("label")
        if start is None or end is None:
            continue
        if label not in (NER_LABEL_CHARACTERISTIC, NER_LABEL_PROPERTY):
            continue
        indices = [
            index for index, word in enumerate(words) if _overlap(word, int(start), int(end))
        ]
        for position, index in enumerate(indices):
            prefix = NER_BIO_PREFIX_BEGIN if position == 0 else NER_BIO_PREFIX_INSIDE
            labels[index] = f"{prefix}{label}"
    return labels


def load_weak_records(path):
    records = []
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def records_to_examples(records):
    examples = []
    for record in tqdm(records, desc="Конвертация в BIO"):
        text = record.get("text") or ""
        words = tokenize_words(text)
        labels = align_labels(words, record.get("spans") or [])
        examples.append(BioExample([word.text for word in words], labels))
    return examples


In [ ]:
from collections import Counter

train_records = load_weak_records(train_path)
val_records = load_weak_records(val_path)
print(f"Записей: train={len(train_records)}, val={len(val_records)}")

train_examples = records_to_examples(train_records)
val_examples = records_to_examples(val_records)


def dataset_stats(examples, name):
    counts = Counter()
    entities = 0
    empty = 0
    lengths = []
    for example in examples:
        lengths.append(len(example.words))
        found = 0
        for label in example.labels:
            counts[label] += 1
            if label.startswith(NER_BIO_PREFIX_BEGIN):
                found += 1
        entities += found
        if found == 0:
            empty += 1
    lengths.sort()
    median = lengths[len(lengths) // 2] if lengths else 0
    p95 = lengths[int(len(lengths) * 0.95)] if lengths else 0
    maximum = lengths[-1] if lengths else 0
    print(f"\n{name}: примеров={len(examples)}, сущностей={entities}, без сущностей={empty}")
    print(f"слов: медиана={median}, p95={p95}, максимум={maximum}")
    print("метки (токены):", dict(counts))


dataset_stats(train_examples, "train")
dataset_stats(val_examples, "val")


## 3. Параметры обучения

Параметры подбираются под доступную GPU автоматически. Если случится OOM —
уменьши `BATCH_SIZE`/`MAX_LENGTH` и перезапусти ячейки с этим блоком.

`TRAIN_SUBSET` — быстрый прогон на подвыборке (0 = все данные). Для проверки
можно поставить 200, убедиться, что всё работает, затем вернуть 0.


In [ ]:
import random

import numpy as np
import torch

MODEL_NAME = "DeepPavlov/rubert-base-cased"
OUTPUT_DIR = "/content/ner_model"
SEED = 42
EPOCHS = 2
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
TRAIN_SUBSET = 0

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if total_gb >= 20:
        BATCH_SIZE, GRAD_ACCUM, MAX_LENGTH = 16, 1, 512
    elif total_gb >= 14:
        BATCH_SIZE, GRAD_ACCUM, MAX_LENGTH = 8, 2, 512
    else:
        BATCH_SIZE, GRAD_ACCUM, MAX_LENGTH = 8, 2, 384
else:
    BATCH_SIZE, GRAD_ACCUM, MAX_LENGTH = 2, 8, 256

print(f"GPU: {'да' if torch.cuda.is_available() else 'нет'}")
print(f"model={MODEL_NAME}")
print(f"batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, max_length={MAX_LENGTH}, epochs={EPOCHS}")
print(f"train_subset={TRAIN_SUBSET or 'все данные'}")


## 4. Модель, датасеты и проверка пайплайна

Сначала выполняется один forward+backward на двух примерах — так ошибки
(несовпадение словаря меток, OOM, кривой батч) всплывут до долгого обучения.


In [ ]:
from torch.utils.data import Dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
)


class BioTorchDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        encoded = self.tokenizer(
            example.words,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
        )
        word_ids = encoded.word_ids()
        aligned = []
        previous = None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)
            elif word_id != previous:
                aligned.append(LABEL2ID[example.labels[word_id]])
            else:
                aligned.append(-100)
            previous = word_id
        encoded["labels"] = aligned
        return encoded


train_examples_used = train_examples[:TRAIN_SUBSET] if TRAIN_SUBSET else train_examples

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
collator = DataCollatorForTokenClassification(tokenizer)

train_dataset = BioTorchDataset(train_examples_used, tokenizer, MAX_LENGTH)
val_dataset = BioTorchDataset(val_examples, tokenizer, MAX_LENGTH)
print(f"train_dataset={len(train_dataset)}, val_dataset={len(val_dataset)}")

sample_batch = collator([train_dataset[i] for i in range(min(2, len(train_dataset)))])
sample_batch = {key: value.to(device) for key, value in sample_batch.items()}
model.train()
outputs = model(**sample_batch)
outputs.loss.backward()
model.zero_grad(set_to_none=True)
print(f"Проверка пайплайна пройдена: loss={outputs.loss.item():.4f}")
if device == "cuda":
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"Пик памяти на одном шаге: {peak:.2f} ГБ")
    torch.cuda.reset_peak_memory_stats()


## 5. Обучение

Прогресс виден по полосе `Trainer` (батчи/эпохи) и логам каждые 25 шагов.
Метрики считаются по сущностям (entity-level), а не по токенам.


In [ ]:
import inspect

from transformers import Trainer, TrainingArguments


def entity_spans(label_ids):
    labels = [ID2LABEL[int(value)] if int(value) >= 0 else NER_BIO_OUTSIDE for value in label_ids]
    entities = set()
    start = None
    entity = None
    for index, label in enumerate(labels):
        if label == NER_BIO_OUTSIDE:
            if entity is not None:
                entities.add((entity, start, index))
            start, entity = None, None
            continue
        if label.startswith(NER_BIO_PREFIX_BEGIN):
            if entity is not None:
                entities.add((entity, start, index))
            start = index
            entity = label[len(NER_BIO_PREFIX_BEGIN) :]
        elif label.startswith(NER_BIO_PREFIX_INSIDE):
            current = label[len(NER_BIO_PREFIX_INSIDE) :]
            if entity != current:
                if entity is not None:
                    entities.add((entity, start, index))
                start = index
                entity = current
    if entity is not None:
        entities.add((entity, start, len(labels)))
    return entities


def scores_from_sets(true_sets, pred_sets):
    tp = fp = fn = 0
    for true, pred in zip(true_sets, pred_sets):
        tp += len(true & pred)
        fp += len(pred - true)
        fn += len(true - pred)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = logits.argmax(axis=-1)
    true_sets = [entity_spans(row) for row in labels]
    pred_sets = [entity_spans(row) for row in predictions]
    return scores_from_sets(true_sets, pred_sets)


training_kwargs = dict(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=max(BATCH_SIZE, 8),
    gradient_accumulation_steps=GRAD_ACCUM,
    weight_decay=WEIGHT_DECAY,
    save_strategy="epoch",
    save_total_limit=1,
    logging_steps=25,
    seed=SEED,
    report_to="none",
    fp16=torch.cuda.is_available(),
    disable_tqdm=False,
)
training_signature = inspect.signature(TrainingArguments.__init__).parameters
eval_key = "eval_strategy" if "eval_strategy" in training_signature else "evaluation_strategy"
training_kwargs[eval_key] = "epoch"
if "warmup_ratio" in training_signature:
    training_kwargs["warmup_ratio"] = WARMUP_RATIO
else:
    # В transformers >= 5 warmup_ratio убран: warmup_steps принимает долю (float).
    training_kwargs["warmup_steps"] = WARMUP_RATIO

trainer_kwargs = dict(
    model=model,
    args=TrainingArguments(**training_kwargs),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer_signature = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)
train_result = trainer.train()
print("Обучение завершено:", train_result.metrics)

eval_metrics = trainer.evaluate()
print("Метрики на val:", eval_metrics)


## 6. Детальные метрики и примеры предсказаний

Entity-level метрики по каждому классу + несколько примеров из val, чтобы
глазами проверить, что модель выделяет.


In [ ]:
predictions_output = trainer.predict(val_dataset)
logits = predictions_output.predictions
label_ids = predictions_output.label_ids
predicted_ids = logits.argmax(axis=-1)

true_sets = [entity_spans(row) for row in label_ids]
pred_sets = [entity_spans(row) for row in predicted_ids]

metrics = {"overall": scores_from_sets(true_sets, pred_sets)}
print("OVERALL:", metrics["overall"])
for entity in (NER_LABEL_CHARACTERISTIC, NER_LABEL_PROPERTY):
    true_subset = [{item for item in spans if item[0] == entity} for spans in true_sets]
    pred_subset = [{item for item in spans if item[0] == entity} for spans in pred_sets]
    metrics[entity] = scores_from_sets(true_subset, pred_subset)
    print(entity + ":", metrics[entity])


def split_label(label):
    if label == NER_BIO_OUTSIDE:
        return NER_BIO_OUTSIDE, None
    for prefix in (NER_BIO_PREFIX_BEGIN, NER_BIO_PREFIX_INSIDE):
        if label.startswith(prefix):
            return prefix, label[len(prefix) :]
    return NER_BIO_OUTSIDE, None


def decode_bio(text, tokens):
    spans = []
    current = None

    def flush():
        nonlocal current
        if current is not None:
            spans.append(dict(current))
            current = None

    for start, end, label, score in tokens:
        if start == end:
            continue
        prefix, entity = split_label(label)
        if entity is None:
            flush()
            continue
        if current is not None and current["label"] == entity and prefix == NER_BIO_PREFIX_INSIDE:
            current["end"] = end
            current["score"] = min(current["score"], score)
            continue
        flush()
        current = {"start": start, "end": end, "label": entity, "score": score}
    flush()
    return spans


model.eval()
for record in val_records[:3]:
    text = record.get("text", "")
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )
    offsets = encoded.pop("offset_mapping")[0].tolist()
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.no_grad():
        logits_one = model(**encoded).logits[0]
    probabilities = torch.softmax(logits_one, dim=-1)
    scores, ids = probabilities.max(dim=-1)
    tokens = [
        (int(start), int(end), ID2LABEL[int(label_id)], float(score))
        for (start, end), label_id, score in zip(offsets, ids.tolist(), scores.tolist())
        if start != end
    ]
    predicted = decode_bio(text, tokens)
    print("=" * 80)
    print("Текст:", text[:300].replace("\n", " "))
    print("Предсказания:")
    for span in predicted[:15]:
        snippet = text[span["start"] : span["end"]]
        print(f"  [{span['label']}] {snippet!r} (score={span['score']:.3f})")


## 7. Сохранение результата

Сохраняются модель, токенизатор, `labels.json` и `metrics.json`. Если смонтирован
Google Drive — архив копируется туда, иначе браузер скачает `ner_model.zip`.

Эти файлы потом кладутся в `models/ner` проекта — `TransformersSpanPredictor`
загружает их без дополнительной конвертации.


In [ ]:
import json
import shutil
from pathlib import Path

output = Path(OUTPUT_DIR)
output.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(output))
tokenizer.save_pretrained(str(output))
(output / "labels.json").write_text(
    json.dumps(LABELS, ensure_ascii=False, indent=2), encoding="utf-8"
)
(output / "metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)

archive_base = str(output.parent / "ner_model")
archive = shutil.make_archive(archive_base, "zip", str(output))
print("Архив:", archive, f"{Path(archive).stat().st_size / 1e6:.1f} МБ")

drive_dir = Path("/content/drive/MyDrive")
if drive_dir.exists():
    target = drive_dir / "ner_model.zip"
    shutil.copy(archive, target)
    print("Скопировано в Google Drive:", target)
else:
    try:
        from google.colab import files

        files.download(archive)
    except Exception as exc:
        print("Скачай архив вручную:", archive, "|", exc)


## Если что-то пошло не так

- **OOM / «CUDA out of memory»** — уменьши `BATCH_SIZE` (8 → 4) и `MAX_LENGTH`
  (512 → 384) в блоке «Параметры обучения», перезапусти ячейки начиная с него.
- **Отключился runtime** — Colab рвёт сессии по таймауту. Запусти с Drive
  (`drive.mount`) до обучения и используй `TRAIN_SUBSET` для быстрой проверки.
- **Метрики низкие** — проверь, что загружен правильный `train.jsonl`
  (weak-датасет из `make_ner_weak_dataset.py`), и что GPU включён.
- **Модель не скачалась** — каталог `ner_model` всё равно лежит в `/content`;
  скачай его через панель файлов слева.
